<a href="https://colab.research.google.com/github/MElsdk-lab/Biochar_forest_estimation/blob/main/Notebook_3_Forest_type_composition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# NOTEBOOK 3 — GEE Forest compposition and Type Computation
# University of Pittsburgh | Biochar Feedstock Methodology
# ============================================================

In [2]:
# ── CELL 1: Install Libraries ─────────────────────────────────────────────────
!pip install -q earthengine-api geemap

In [3]:
# ── CELL 2: conect to drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
DRIVE_FOLDER = '/content/drive/MyDrive/BiocharProject/'
print('✅ connected to drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ connected to drive


In [4]:
# ── CELL 3: clone repo if not already cloned ─────────────────────
import os
import getpass
import subprocess


if not os.path.exists('/content/Biochar_forest_estimation'):
    PAT = getpass.getpass('Enter PAT: ')
    !git config --global user.email "sdkmajd@gmail.com"
    !git config --global user.name "MElsdk-lab"
    subprocess.run(
        f'git clone https://{PAT}@github.com/MElsdk-lab/Biochar_forest_estimation.git',
        shell=True
    )

%cd /content/Biochar_forest_estimation/
!git fetch origin
!git reset --hard origin/main
print('✅ Ready')

/content/Biochar_forest_estimation
HEAD is now at 7cbdf8c Ada comments to explain
✅ Ready


In [5]:
# ── CELL 4: import libraries and data from data_config ─────────────────────

import ee
import geemap
import pandas as pd
import time
from data_config import FAO_LSIB_REGION, us_state_names, get_all_countries, build_country_lookup, forest_bins

print('✅ Libraries imported')
print('✅ Data config loaded')

✅ Libraries imported
✅ Data config loaded


In [6]:
# ── CELL 5: Authenticate and initialize Earth Engine & Load Datasets  ─────────────────────
try:
    ee.Initialize(project='spry-blade-487218-n0')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='spry-blade-487218-n0')  # ← add project here too

Hansen_GFC_2024      = ee.Image("UMD/hansen/global_forest_change_2024_v1_12")
GLC_FSC30D_annual    = ee.ImageCollection('projects/sat-io/open-datasets/GLC-FCS30D/annual')
GLC_FSC30D_five_year = ee.ImageCollection('projects/sat-io/open-datasets/GLC-FCS30D/five-years-map')

print('✅ Hansen GFC 2024 loaded')
print('✅ GLC FCS30D annual loaded')
print('✅ GLC FCS30D five-year loaded')


✅ Hansen GFC 2024 loaded
✅ GLC FCS30D annual loaded
✅ GLC FCS30D five-year loaded


In [7]:
# ── CELL 6: processing datasets ───────────────────────────────────────

# Select & Mask Hansen Bands
treecover2000 = Hansen_GFC_2024.select('treecover2000')
datamask      = Hansen_GFC_2024.select('datamask')

treecover2000_masked = treecover2000.updateMask(treecover2000.gt(0)) \
                                    .updateMask(datamask.eq(1))

print('✅ treecover2000 masked')

#Load GLC FCS30D Year 2000

glc_2000        = GLC_FSC30D_annual.mosaic().select('b1')
glc_2000_forest = glc_2000.gte(51).And(glc_2000.lte(92)).selfMask()

print('✅ GLC FCS30D 2000 loaded')
print('✅ GLC forest mask created (classes 51-92)')

# Forest Class Definitions
forestClasses = [
    {'code': 51, 'color': '4c7300', 'name': '51 Open evergreen broadleaved'},
    {'code': 52, 'color': '006400', 'name': '52 Closed evergreen broadleaved'},
    {'code': 61, 'color': 'a8c800', 'name': '61 Open deciduous broadleaved'},
    {'code': 62, 'color': '00a000', 'name': '62 Closed deciduous broadleaved'},
    {'code': 71, 'color': '005000', 'name': '71 Open evergreen needleleaved'},
    {'code': 72, 'color': '003c00', 'name': '72 Closed evergreen needleleaved'},
    {'code': 81, 'color': '286400', 'name': '81 Open deciduous needleleaved'},
    {'code': 82, 'color': '285000', 'name': '82 Closed deciduous needleleaved'},
    {'code': 91, 'color': 'a0b432', 'name': '91 Open mixed forest'},
    {'code': 92, 'color': '788200', 'name': '92 Closed mixed forest'},
]

print(f'✅ {len(forestClasses)} forest classes defined')

✅ treecover2000 masked
✅ GLC FCS30D 2000 loaded
✅ GLC forest mask created (classes 51-92)
✅ 10 forest classes defined


In [8]:
# ── CELL 7: GEE Functions — Forest Cover by Bin ──────────────────────────────

def get_forest_cover_bins_one_country(country_feature, bins):
    """
    For one country feature, compute forest area (ha) per canopy cover bin.
    Returns a GEE FeatureCollection — one Feature per bin.
    """
    all_features = ee.FeatureCollection([])

    for i in range(len(bins) - 1):
        bin_label = f'{bins[i]}-{bins[i+1]}'

        forest_mask_bin = (
            treecover2000_masked.gte(bins[i])
            .And(treecover2000_masked.lt(bins[i+1]))
            .selfMask()
            .updateMask(datamask.eq(1))
        )

        area_image = forest_mask_bin.multiply(ee.Image.pixelArea().divide(1e4))

        region_area = area_image.reduceRegions(
            collection=country_feature.geometry(),
            reducer=ee.Reducer.sum(),
            scale=30
        )

        region_area = region_area.map(lambda f: f.set('bin', bin_label))
        all_features = all_features.merge(region_area)

    return all_features


def export_forest_cover_bins_all_countries(selected_regions, bins):
    """
    Loop over all countries, build one combined FeatureCollection,
    and submit a single export task to Drive.
    """
    all_countries = get_all_countries(selected_regions)

    lsib_fao = ee.FeatureCollection('USDOS/LSIB_SIMPLE/2017') \
                  .filter(ee.Filter.inList('country_na', all_countries))

    country_list = lsib_fao.toList(lsib_fao.size())
    n = lsib_fao.size().getInfo()

    combined = ee.FeatureCollection([])

    for i in range(n):
        country_feature = ee.Feature(country_list.get(i))
        fc = get_forest_cover_bins_one_country(country_feature, bins)
        combined = combined.merge(fc)

    task = ee.batch.Export.table.toDrive(
        collection=combined,
        description='forest_cover_bins_all_countries',
        folder='GEE_exports',
        fileNamePrefix='forest_cover_bins_all_countries',
        fileFormat='CSV',
        selectors=['country_na', 'bin', 'sum']
    )
    task.start()
    print('✅ Single export task submitted: forest_cover_bins_all_countries')
    return task

In [9]:
country_forest_area_bins = export_forest_cover_bins_all_countries(FAO_LSIB_REGION, forest_bins)

✅ Single export task submitted: forest_cover_bins_all_countries


In [ ]:
import time

while True:
    current_status = country_forest_area_bins.status()['state']
    print(current_status)
    if current_status in ['COMPLETED', 'FAILED']:
        print('Task finished.')
        break
    time.sleep(150)
    print('waiting...')

READY
waiting...
READY
waiting...
READY
waiting...
READY
waiting...
READY
waiting...
READY
waiting...
READY


In [ ]:


def export_forest_area(selected_regions, thresholds):
    """Submit one export task per threshold to Google Drive."""
    tasks = []
    for threshold in thresholds:
        fc = prepare_forest_collection(selected_regions, threshold)
        # Configure GEE export task to Drive
        task = ee.batch.Export.table.toDrive(
            collection=fc,
            description=f'forest_area_{threshold}',
            folder='GEE_exports',
            fileNamePrefix=f'forest_area_{threshold}',
            fileFormat='CSV',
            selectors=['country_na', 'threshold', 'sum']
        )
        task.start()
        tasks.append(task)
        print(f'✅ Task submitted for threshold {threshold}%')
    return tasks

print('✅ prepare_forest_collection() defined')
print('✅ export_forest_area() defined')

In [ ]:
# ── CELL 12: Push results to GitHub ────────────────────────────────
import subprocess

# ask for PAT only if not already defined
if 'PAT' not in globals():
    import getpass
    PAT = getpass.getpass('Enter PAT: ')

%cd /content/Biochar_forest_estimation/
!git add .
!git commit -m "update results"

result = subprocess.run(
    f'git push https://{PAT}@github.com/MElsdk-lab/Biochar_forest_estimation.git main',
    shell=True,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print('✅ Pushed to GitHub')
else:
    print('❌ Push failed')
    print(result.stderr)